# SafeQuery on Colab (Run 6) - full retrain from GitHub: base → SFT → GRPO → benchmarks → ROI
**Lexsi Labs hackathon - Agentic Fine-tuning track** - AgentTune (training) + AuditKIT (evaluation) on **Tiny Aya Global**.

Run 6 is the **reproducibility run**: nothing is restored from Drive and nothing is uploaded by hand.
1. Clones the code from **github.com/siva2k16/LLMHackathon2026** and installs AgentTune + AuditKIT (unmodified).
2. Rebuilds the database and tasks (seeds 7 / 1 / 2) and checks them against the Run 5 / Hugging Face copies by md5.
3. Evaluates the **base** model, **trains SFT from scratch**, evaluates it, merges it, **trains GRPO** with the Run 5 settings, evaluates it.
4. Runs BFCL + ARC for base, SFT **and GRPO** (cache cleared per model), writes the comparison table and ROI.

Results go to **`results_run6/`** (the repo's `results/` holds Run 5 and is never touched) and to **`MyDrive/safequery_run6`** after every long step.
Every long step is **skip-if-done**, so after a disconnect: reconnect, run the Setup + Config cells, run **Step R**, and continue.

**Runtime:** A100 (recommended). Rough times on an A100: install 5-15 min · base eval ~25 min · SFT ~40-60 min · SFT eval ~25 min · GRPO ~1-2 h · GRPO eval ~25 min · benchmarks ~20 min per model.

In [ ]:
!nvidia-smi --query-gpu=name,compute_cap,memory.total --format=csv
import sys; print(sys.version)  # AgentTune needs Python >= 3.12

## 0 - Get the code from GitHub

In [ ]:
import os
PROJ = "/content/LLMHackathon2026"
REPO_URL = "https://github.com/siva2k16/LLMHackathon2026"
if not os.path.exists(f"{PROJ}/scripts/build_data.py"):
    !git clone -q {REPO_URL} {PROJ}
else:
    !git -C {PROJ} pull -q
%cd {PROJ}
!git log --oneline -1
!grep -c "def trim_turn" safequery/policies.py && grep -c "CallableModel" scripts/eval_benchmarks.py && echo "fixes A-C present"

## 1 - Install AgentTune and AuditKIT (unmodified, from Lexsi Labs)
5-15 minutes. The red **dependency conflict** warnings are harmless.
**After it installs, use Runtime -> Restart session**, then continue from the next cell (the code stays on disk).

In [ ]:
!git clone -q https://github.com/Lexsi-Labs/AgentTune.git /content/AgentTune 2>/dev/null || echo "AgentTune already cloned"
!git clone -q https://github.com/Lexsi-Labs/AuditKIT.git /content/AuditKIT 2>/dev/null || echo "AuditKIT already cloned"
import subprocess
if subprocess.run(["python", "-c", "import agenttune, auditkit"], capture_output=True).returncode == 0:
    print("AgentTune + AuditKIT already installed - skipping")
else:
    !pip install -q --only-binary=pandas -e /content/AgentTune -e /content/AuditKIT
    print("Installed. Now: Runtime -> Restart session, then run the next cells.")

In [ ]:
# Lets this kernel import the editable installs even if you didn't restart
import sys
for p in ["/content/AgentTune/src", "/content/AuditKIT/src", "/content/AuditKIT"]:
    if p not in sys.path: sys.path.insert(0, p)
import torch, transformers, trl, peft, agenttune, auditkit
print("torch", torch.__version__, "| cuda", torch.cuda.is_available(),
      "| native bf16", torch.cuda.is_available() and torch.cuda.is_bf16_supported(including_emulation=False))
print("transformers", transformers.__version__, "| trl", trl.__version__, "| peft", peft.__version__)

## 1b - Hugging Face login
Tiny Aya is **gated**: accept the terms at https://huggingface.co/CohereLabs/tiny-aya-global first.
Use a **new** fine-grained token with read access (write access only if you will push at the end). Never paste tokens into code cells.

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

## Config - **re-run this cell after every restart**

In [ ]:
import os, glob, shutil, hashlib, json, time, torch
PROJ = "/content/LLMHackathon2026"
%cd {PROJ}

MODEL     = "CohereLabs/tiny-aya-global"
RES       = "results_run6"          # Run 6 output folder (the repo's results/ = Run 5, untouched)
LIMIT     = None                    # None = all test tasks
LOAD_4BIT = False                   # True only if an eval runs out of GPU memory
RUN_BASE  = True                    # evaluate the base model again
RUN_GRPO  = True
# Run 5 GRPO settings (kept identical so Run 6 reproduces Run 5)
GRPO_ARGS = "--max-steps 60 --num-generations 2 --batch-size 2 --max-completion-length 192"
HF_USER   = "Siva2022"

LIMIT_ARG = f"--limit {LIMIT}" if LIMIT else ""
Q_ARG     = "--load-4bit" if LOAD_4BIT else ""
DRIVE_DIR = "/content/drive/MyDrive/safequery_run6"
GPU_NAME  = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
os.makedirs(RES, exist_ok=True)

# md5 of the Run 5 data files (also published in the safequery-retail-tasks dataset)
RUN5_MD5 = {
    "tasks_train.jsonl": "1297c43b4683d016f9529620a41bb0b9",
    "tasks_test.jsonl":  "aa5cd9fac7e2f137f52974e1d6a0643a",
    "sft_train.jsonl":   "ab308961f042e5cefe54205c1eef60df",
    "grpo_train.jsonl":  "e38fa26dcbf4e7c5a8815d4c9eaf99c0",
}

def _mount():
    from google.colab import drive
    if not os.path.ismount("/content/drive"):
        drive.mount("/content/drive")

def _md5(p):
    return hashlib.md5(open(p, "rb").read()).hexdigest() if os.path.exists(p) else None

def _done(run):
    p = f"{RES}/{run}/metrics.json"
    return os.path.exists(p) and (LIMIT or json.load(open(p))["metrics"]["n_tasks"] >= (n_test or 0))

def save_to_drive():
    """Copy results_run6/, runs/ (minus merged model + checkpoints) and data jsonl to Drive."""
    _mount(); os.makedirs(DRIVE_DIR, exist_ok=True)
    skip = shutil.ignore_patterns("sft_merged", "checkpoint-*")
    for d in [RES, "runs"]:
        if os.path.exists(d):
            shutil.copytree(d, f"{DRIVE_DIR}/{d}", dirs_exist_ok=True, ignore=skip)
    os.makedirs(f"{DRIVE_DIR}/data", exist_ok=True)
    for f in glob.glob("data/*.jsonl"):
        shutil.copy(f, f"{DRIVE_DIR}/data")
    print("saved ->", DRIVE_DIR, sorted(os.listdir(DRIVE_DIR)))

TIMES = {}
def timed(label, t0):
    TIMES[label] = round((time.time() - t0) / 60, 1); print(f"[time] {label}: {TIMES[label]} min")

n_test = sum(1 for _ in open("data/tasks_test.jsonl")) if os.path.exists("data/tasks_test.jsonl") else None
print("GPU:", GPU_NAME, "| cwd", os.getcwd(), "| test tasks:", n_test or "- (run step 2)")

## R - *Only after a disconnect*: restore Run 6 progress from Drive
Skip this on the first run. It copies `results_run6/` and `runs/` back from `MyDrive/safequery_run6`, so finished steps are skipped.

In [ ]:
RESTORE = False   # set True after a disconnect
if RESTORE:
    _mount()
    for d in [RES, "runs"]:
        if os.path.exists(f"{DRIVE_DIR}/{d}"):
            shutil.copytree(f"{DRIVE_DIR}/{d}", d, dirs_exist_ok=True); print("restored", d)
else:
    print("RESTORE is False - fresh run")

## 2 - Build the database and tasks, check they match Run 5
Seeds: database 7, train tasks 1, test tasks 2. The files must match Run 5 exactly, so Run 6 numbers are on the same tasks.

In [ ]:
%cd {PROJ}
!python scripts/build_data.py
print()
ok = True
for f, want in RUN5_MD5.items():
    got = _md5(f"data/{f}")
    print(f"{f:20s}", "MATCH" if got == want else f"DIFFERENT ({got})")
    ok &= got == want
n_test = sum(1 for _ in open("data/tasks_test.jsonl"))
print("test tasks:", n_test, "| all match Run 5:", ok)

In [ ]:
%cd {PROJ}
!python scripts/smoke_test.py --results {RES}

## 3 - Baseline: Tiny Aya Global, no training

In [ ]:
%cd {PROJ}
if not RUN_BASE:
    print("RUN_BASE is False - skipping")
elif _done("base"):
    print("base already evaluated - skipping (delete", f"{RES}/base", "to re-run)")
else:
    t0 = time.time()
    !python scripts/eval_model.py --name base --model {MODEL} --results {RES} {LIMIT_ARG} {Q_ARG}
    timed("base eval", t0)
save_to_drive()

## 4 - Train SFT from scratch (AgentTune, LoRA r16)
Defaults = Run 3/5 settings: 2 epochs, lr 2e-4, batch 1 × 8 accumulation, max length 2048, LoRA r16 / α32 / dropout 0.05.

In [ ]:
%cd {PROJ}
if os.path.exists("runs/sft/adapter_model.safetensors"):
    print("SFT adapter already exists - skipping (delete runs/sft to retrain)")
else:
    t0 = time.time()
    !python scripts/train_sft.py --model {MODEL} --out runs/sft --epochs 2
    timed("SFT train", t0)
log = "runs/sft/safequery_train_log.json"
if os.path.exists(log):
    L = json.load(open(log)); print({k: L[k] for k in list(L)[:8]})
save_to_drive()

## 5 - Evaluate SFT on all test tasks

In [ ]:
%cd {PROJ}
assert os.path.exists("runs/sft/adapter_config.json"), "no SFT adapter - run step 4"
if _done("sft"):
    print("SFT already evaluated - skipping")
else:
    t0 = time.time()
    !python scripts/eval_model.py --name sft --model {MODEL} --adapter runs/sft --results {RES} {LIMIT_ARG} {Q_ARG}
    timed("SFT eval", t0)
save_to_drive()

## 6 - GRPO with AgentTune (from the merged SFT model)
Same settings as Run 5: 60 steps, 2 generations per prompt, batch 2, max completion 192 tokens (lr 5e-6, β 0.02, temperature 0.9 are the script defaults).

In [ ]:
%cd {PROJ}
if not RUN_GRPO:
    print("RUN_GRPO is False - skipping")
else:
    if not os.path.exists("runs/sft_merged/config.json"):
        t0 = time.time()
        !python scripts/merge_adapter.py --model {MODEL} --adapter runs/sft --out runs/sft_merged
        timed("merge", t0)
    if os.path.exists("runs/grpo/adapter_model.safetensors"):
        print("GRPO adapter already exists - skipping (delete runs/grpo to retrain)")
    else:
        t0 = time.time()
        !python scripts/train_grpo.py --model runs/sft_merged --out runs/grpo {GRPO_ARGS}
        timed("GRPO train", t0)
    save_to_drive()

In [ ]:
%cd {PROJ}
if RUN_GRPO:
    if _done("grpo"):
        print("GRPO already evaluated - skipping")
    else:
        t0 = time.time()
        !python scripts/eval_model.py --name grpo --model runs/sft_merged --adapter runs/grpo --results {RES} {LIMIT_ARG} {Q_ARG}
        timed("GRPO eval", t0)
    save_to_drive()

## 7 - Public benchmarks via AuditKIT: BFCL (tool calling) + ARC
The AuditKIT disk cache is cleared before each model, and each run must show **model calls > 0** (Run 3's cache bug).

In [ ]:
%cd {PROJ}
AK_CACHE = os.path.join(os.environ.get("XDG_CACHE_HOME", os.path.expanduser("~/.cache")), "auditkit", "runs")
BENCH = [("base", f"--model {MODEL}"), ("sft", f"--model {MODEL} --adapter runs/sft")]
if RUN_GRPO:
    BENCH.append(("grpo", "--model runs/sft_merged --adapter runs/grpo"))
for name, args in BENCH:
    out = f"{RES}/{name}/benchmarks.json"
    if os.path.exists(out):
        print(name, "benchmarks already done - skipping"); continue
    shutil.rmtree(AK_CACHE, ignore_errors=True)
    t0 = time.time()
    !python scripts/eval_benchmarks.py --name {name} {args} --results {RES} --bfcl-limit 100 --arc-limit 200 {Q_ARG}
    timed(f"{name} benchmarks", t0)
    b = json.load(open(out)); calls = b["throughput"]["model_calls"]
    print(f"[check] {name}: {calls} model calls", "OK" if calls > 0 else "<- CACHED, numbers not valid")
    save_to_drive()

## 8 - Comparison table, Run 5 vs Run 6, and ROI

In [ ]:
%cd {PROJ}
GPU_PRICE = 0.35   # $/h - update for the GPU you ran on (check the current price page)
RUNS = ("base " if os.path.exists(f"{RES}/base/metrics.json") else "") + "sft" + (" grpo" if RUN_GRPO else "")
!python scripts/compare.py --results {RES} --runs {RUNS} --out {RES}/COMPARISON.md
!python scripts/roi_report.py --run {RES}/sft --closed-in 3.0 --closed-out 15.0 --gpu-name "{GPU_NAME}" --gpu-price {GPU_PRICE} --out {RES}/ROI.md
from IPython.display import Markdown, display
display(Markdown(open(f"{RES}/COMPARISON.md", encoding="utf-8").read()))
display(Markdown(open(f"{RES}/ROI.md", encoding="utf-8").read()))

# Reproducibility check: Run 6 vs the Run 5 numbers stored in the repo (results/)
keys = ["task_success_rate", "answer_accuracy", "refusal_accuracy", "injection_resistance", "intent_accuracy", "unsafe_attempt_rate", "invalid_call_rate"]
rows = []
for run in RUNS.split():
    p5, p6 = f"results/{run}/metrics.json", f"{RES}/{run}/metrics.json"
    if os.path.exists(p5) and os.path.exists(p6):
        m5, m6 = json.load(open(p5))["metrics"], json.load(open(p6))["metrics"]
        rows.append("| " + run + " | " + " | ".join(f"{m5.get(k, float('nan')):.3f} → {m6.get(k, float('nan')):.3f}" for k in keys) + " |")
display(Markdown("| run | " + " | ".join(keys) + " |\n|" + "---|" * (len(keys) + 1) + "\n" + "\n".join(rows)))
json.dump({"gpu": GPU_NAME, "minutes": TIMES}, open(f"{RES}/run6_timing.json", "w"), indent=2)
print("timings (min):", TIMES)
save_to_drive()

## 9 - Download results
`safequery_results_run6.zip` holds `results_run6/`, the data files and the SFT/GRPO training logs (no model weights).

In [ ]:
%cd {PROJ}
!zip -r -q /content/safequery_results_run6.zip {RES} data/*.jsonl runs/sft/safequery_train_log.json runs/grpo/*.json 2>/dev/null
!ls -lh /content/safequery_results_run6.zip
from google.colab import files
files.download("/content/safequery_results_run6.zip")

## 10 - *Optional*: publish Run 6 to Hugging Face
**Off by default.** Pushing would **overwrite** the Run 5 adapters and traces in the existing public repos (same repo names).
Only set `PUSH = True` if you decide Run 6 replaces Run 5. Needs a token with write access.

In [ ]:
%cd {PROJ}
PUSH = False
ADAPTERS = "sft=runs/sft" + (" grpo=runs/grpo" if RUN_GRPO else "")
if PUSH and HF_USER:
    !python scripts/push_to_hub.py --user {HF_USER} --adapters {ADAPTERS} --results {RES}
else:
    print("PUSH is False - skipping (Run 5 artifacts on Hugging Face stay as they are)")